In [ ]:
from pathlib import Path

input_root = Path("/kaggle/input")

print("Kaggle input datasets:")
for item in input_root.iterdir():
    print(f"{'DIR ' if item.is_dir() else 'FILE'} {item}")

In [ ]:
from pathlib import Path

COMP_DIR = next(Path("/kaggle/input/competitions").iterdir())

print(f"Competition directory: {COMP_DIR}")
print("\nTop-level contents:")

for item in COMP_DIR.iterdir():
    print(f"{'DIR ' if item.is_dir() else 'FILE'}  {item.name}")

In [ ]:
import pandas as pd

# Discover and load all CSV files
csv_files = {
    file.stem: file
    for file in COMP_DIR.iterdir()
    if file.is_file() and file.suffix.lower() == ".csv"
}

print("Discovered CSV files:")
for name in sorted(csv_files):
    print(f"  {name}.csv")

dataframes = {
    name: pd.read_csv(path)
    for name, path in csv_files.items()
}

print("\n===== DATASET SHAPES =====")
for name, df in dataframes.items():
    print(f"{name:20s}: {df.shape}")

In [ ]:
print("===== TRAIN =====")

train_df = next(
    df for df in dataframes.values()
    if "Report" in df.columns
    and "StudyInstanceUID" in df.columns
)

display(train_df.head())
print("Columns:", train_df.columns.tolist())


print("\n===== TRAIN SERIES =====")

series_candidates = [
    df for df in dataframes.values()
    if {
        "StudyInstanceUID",
        "SeriesInstanceUID",
        "Fluid_Sensitive",
        "Anatomical_Plane"
    }.issubset(df.columns)
]

train_series = max(series_candidates, key=len)

display(train_series.head())
print("Columns:", train_series.columns.tolist())


print("\n===== TEST =====")

test_df = next(
    df for df in dataframes.values()
    if list(df.columns) == ["StudyInstanceUID"]
)

display(test_df.head())
print("Columns:", test_df.columns.tolist())


print("\n===== SAMPLE SUBMISSION =====")

sample_submission = next(
    df for df in dataframes.values()
    if "StudyInstanceUID" in df.columns
    and len(df.columns) == 13
)

display(sample_submission.head())
print("Columns:", sample_submission.columns.tolist())

In [ ]:
TARGET_COLS = [
    col for col in sample_submission.columns
    if col != "StudyInstanceUID"
]

print(f"Number of targets: {len(TARGET_COLS)}")
print("\nTargets:")
for i, col in enumerate(TARGET_COLS, 1):
    print(f"{i:2}. {col}")

In [ ]:
print("===== LABEL AVAILABILITY =====")

label_counts = train_df[TARGET_COLS].notna().sum()

display(
    pd.DataFrame({
        "Labeled": label_counts,
        "Missing": len(train_df) - label_counts
    })
)

In [ ]:
labeled_df = train_df[train_df[TARGET_COLS].notna().all(axis=1)].copy()

label_summary = pd.DataFrame({
    "Positive": labeled_df[TARGET_COLS].sum(),
    "Negative": (1 - labeled_df[TARGET_COLS]).sum(),
    "Positive %": labeled_df[TARGET_COLS].mean() * 100
}).round(2)

display(label_summary)

In [ ]:
print("Labeled studies:", len(labeled_df))

display(
    labeled_df[["StudyInstanceUID"] + TARGET_COLS].reset_index(drop=True)
)

In [ ]:
# Correlation between the 12 abnormalities
label_corr = labeled_df[TARGET_COLS].corr()

display(label_corr.round(2))

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(12, 9))

sns.heatmap(
    label_corr,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    square=True
)

plt.title("Correlation Between Abnormality Labels")
plt.tight_layout()
plt.show()

In [ ]:
# Find the dataframe containing series-level information
series_df = next(
    df for df in dataframes.values()
    if {"StudyInstanceUID", "SeriesInstanceUID"}.issubset(df.columns)
)

# Number of series per study
series_per_study = series_df.groupby("StudyInstanceUID").size()

print("Series per study:")
display(series_per_study.describe().round(2))

print("\nMost common series counts:")
display(series_per_study.value_counts().sort_index().head(20))

In [ ]:
series_candidates = [
    df for df in dataframes.values()
    if {"StudyInstanceUID", "SeriesInstanceUID"}.issubset(df.columns)
]

for i, df in enumerate(series_candidates):
    print(f"Candidate {i}: {df.shape}")


In [ ]:
print("Selected series dataframe:", series_df.shape)

print("\nColumns:")
for col in series_df.columns:
    print(f" - {col}")

print("\nUnique values:")
for col in ["Fluid_Sensitive", "Fat_Suppression", "Anatomical_Plane"]:
    print(f"\n{col}:")
    display(series_df[col].value_counts(dropna=False))

In [ ]:
print(
    "Fluid_Sensitive == Fat_Suppression:",
    (series_df["Fluid_Sensitive"] == series_df["Fat_Suppression"]).all()
)

print("\nUnique combinations:")
display(
    series_df[
        ["Fluid_Sensitive", "Fat_Suppression"]
    ].value_counts().rename("Count").reset_index()
)

In [ ]:
display(
    pd.crosstab(
        series_df["Anatomical_Plane"],
        series_df["Fluid_Sensitive"],
        normalize="index"
    ).round(3)
)

In [ ]:
study_plane_counts = pd.crosstab(
    series_df["StudyInstanceUID"],
    series_df["Anatomical_Plane"]
)

display(study_plane_counts.describe().round(2))

In [ ]:
series_counts = series_df.groupby("StudyInstanceUID").size()

display(series_counts.describe().round(2))

In [ ]:
display(
    pd.crosstab(
        series_df["Anatomical_Plane"],
        series_df["Fluid_Sensitive"]
    )
)

In [ ]:
import pydicom
import matplotlib.pyplot as plt

# Pick the first fully labeled study
study_id = labeled_df["StudyInstanceUID"].iloc[0]

# Find its series
study_series = series_df[
    series_df["StudyInstanceUID"] == study_id
]

display(study_series)

In [ ]:
# Find directories inside the competition folder that contain DICOM files
dcm_dirs = []

for item in COMP_DIR.iterdir():
    if item.is_dir():
        dcm_dirs.append(item)

print("Top-level directories:")
for d in dcm_dirs:
    print(d)

In [ ]:
# Find the training DICOM directory dynamically
train_dcm_dir = next(
    d for d in dcm_dirs
    if d.name == "train_series"
)

# Find the selected study directory
study_dir = next(
    d for d in train_dcm_dir.iterdir()
    if d.name == study_id
)

print("Study directory:", study_dir)

# Show its series directories
series_dirs = [d for d in study_dir.iterdir() if d.is_dir()]

print("Number of series folders:", len(series_dirs))
for d in series_dirs:
    print(d.name)

In [ ]:
for series_dir in series_dirs:
    dcm_files = [f for f in series_dir.iterdir() if f.is_file()]
    print(f"{series_dir.name}: {len(dcm_files)} DICOM files")

In [ ]:
import pydicom
import matplotlib.pyplot as plt

# Pick the first series
selected_series = series_dirs[0]

# Get DICOM files dynamically
dcm_files = sorted(
    [f for f in selected_series.iterdir() if f.is_file()]
)

# Read 3 slices: beginning, middle, end
indices = [0, len(dcm_files)//2, len(dcm_files)-1]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, idx in zip(axes, indices):
    ds = pydicom.dcmread(dcm_files[idx])
    ax.imshow(ds.pixel_array, cmap="gray")
    ax.set_title(f"Slice {idx}")
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
for idx in [0, len(dcm_files)//2, len(dcm_files)-1]:
    ds = pydicom.dcmread(dcm_files[idx])
    print(
        f"File index: {idx} | "
        f"InstanceNumber: {getattr(ds, 'InstanceNumber', 'N/A')} | "
        f"Rows: {ds.Rows} | Columns: {ds.Columns}"
    )

In [ ]:
# Read all slices and inspect their InstanceNumber
slice_info = []

for f in dcm_files:
    ds = pydicom.dcmread(f, stop_before_pixels=True)
    slice_info.append({
        "file": f,
        "InstanceNumber": getattr(ds, "InstanceNumber", None)
    })

slice_info_df = pd.DataFrame(slice_info)

display(
    slice_info_df.sort_values("InstanceNumber").reset_index(drop=True)
)

In [ ]:
ds = pydicom.dcmread(slice_info_df.iloc[len(slice_info_df)//2]["file"])

pixels = ds.pixel_array

print("Shape:", pixels.shape)
print("Dtype:", pixels.dtype)
print("Min:", pixels.min())
print("Max:", pixels.max())
print("Mean:", pixels.mean())
print("Std:", pixels.std())

In [ ]:
img = pixels.astype("float32")

# Per-image min-max normalization
img_norm = (img - img.min()) / (img.max() - img.min() + 1e-8)

fig, axes = plt.subplots(1, 2, figsize=(10, 5))

axes[0].imshow(img, cmap="gray")
axes[0].set_title("Original")
axes[0].axis("off")

axes[1].imshow(img_norm, cmap="gray")
axes[1].set_title("Min-Max Normalized")
axes[1].axis("off")

plt.tight_layout()
plt.show()